# KolektorSDD1 SegFormer-B0 Side-Tuning

This notebook trains a side-tuning variant of SegFormer-B0 for binary crack segmentation on KolektorSDD1. It keeps the same dataset preparation, augmentation cache, loss, metrics, threshold sweep, and preview flow as the SegFormer fine-tuning notebook so results remain comparable.


## Dependency Setup
Install the packages needed for SegFormer training and notebook visualisation.


In [ ]:
!pip install -q transformers accelerate torch torchvision pillow matplotlib tqdm scipy


## Google Drive Mount
Mount Google Drive so the notebook can read KSDD1 data and save experiment artifacts.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Runtime Paths
Set the Colab paths, Drive/local dataset locations, and output directory. Keeping the dataset on local disk is much faster than reading every image from Drive during training.


In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import torch

PROJECT_ROOT = Path("/content")
DRIVE_DATA_ROOT = Path("/content/drive/MyDrive/KolektorSDD-boxes")
LOCAL_DATA_ROOT = Path("/content/KolektorSDD-boxes")
COPY_DATA_TO_LOCAL = True  # Faster than reading every image/mask from Google Drive each epoch.
DATA_ROOT = LOCAL_DATA_ROOT if COPY_DATA_TO_LOCAL else DRIVE_DATA_ROOT
IMG_HEIGHT = 1408
IMG_WIDTH = 512
IMG_SIZE = (IMG_HEIGHT, IMG_WIDTH)
OUTPUT_DIR = PROJECT_ROOT / "segformer_ksdd1_side_tuning_runs"


## Preprocessing Cache
Control whether training augmentations are precomputed. Reusing the cache avoids paying the elastic/noise augmentation cost every run.


In [ ]:
USE_PREPROCESSED_TRAIN = True
PREPROCESS_CACHE_ROOT = Path("/content/ksdd1_segformer_aug_cache")
DRIVE_PREPROCESS_CACHE_ZIP = Path("/content/drive/MyDrive/ksdd1_segformer_aug_cache.zip")
COPY_PREPROCESS_CACHE_FROM_DRIVE = True
AUGMENTED_COPIES_PER_SAMPLE = 24
CACHE_COPIES_PER_EPOCH = 2
FORCE_REBUILD_PREPROCESS_CACHE = False


## Training Defaults
These are the model, optimizer, metric, and early stopping defaults used by all strategies.


In [ ]:
BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
NUM_EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 1e-3
GRAD_CLIP_NORM = 1.0
THR = 0.90
DEFAULT_MIN_AREA = 2000
SELECTION_METRIC = "composite"
SCHEDULER_METRIC = "composite"
IMAGE_FPR_PENALTY = 0.2
PIXEL_FPR_PENALTY = 0.0
ELASTIC_PROB = 0.3
ELASTIC_ALPHA = 25.0
ELASTIC_SIGMA = 5.0
GAUSSIAN_NOISE_PROB = 0.2
GAUSSIAN_NOISE_STD = 0.01
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MIN_DELTA = 1e-4
LR_SCHEDULER = "plateau"
COSINE_T_MAX = 30
PLATEAU_FACTOR = 0.5
PLATEAU_PATIENCE = 2
PLATEAU_MIN_LR = 1e-6
SEED = 42
REQUIRE_CUDA = True
NUM_WORKERS = 0
MODEL_NAME = "nvidia/segformer-b0-finetuned-ade-512-512"


## Side-Tuning Options
Set the side-branch and base-checkpoint options here. Choose the actual strategy in the run cell below: `side_tuning` or `side_tuning_last_stage`.


In [ ]:
AVAILABLE_STRATEGIES = ("side_tuning", "side_tuning_last_stage")

BASE_CHECKPOINT_PATH = "/content/drive/MyDrive/best_ksdd2_segformer_full_ft.pth"
FREEZE_BASE_MODEL = True

SIDE_WIDTH = 32
SIDE_DROPOUT = 0.10
SIDE_INIT_MIX = 0.2
SIDE_LR_MULT = 1.0
MIX_LR_MULT = 5.0
BASE_LR_MULT = 0.1


## Evaluation Settings
These options control train metric reporting and the threshold/min-area sweep after training.


In [ ]:
TRACK_TRAIN_METRICS = False
EVAL_TRAIN_METRICS = False
SYNC_TIMING = False
RUN_THRESHOLD_SWEEP = True
THRESHOLD_SWEEP = "0.5,0.6,0.7,0.75,0.8,0.85,0.88,0.9,0.92,0.95,0.97,0.975,0.98,0.985,0.99"
THRESHOLD_MIN_AREAS = (0, 25, 50, 100, 200, 500, 1000, 1500, 2000, 2500, 3000, 4000)


## Environment Check
Print the active runtime, dataset path, output directory, and available strategy choices before the heavier cells run.


In [ ]:
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Dataset:", DATA_ROOT)
print("Output:", OUTPUT_DIR)
print("Available strategies:", ", ".join(AVAILABLE_STRATEGIES))


## Implementation Overview
The following cells define the reusable pieces of the side-tuning experiment: dataset handling, augmentation, frozen-base and side-branch construction, losses, metrics, training, and reporting. They do not start training until the final run cell.


## Imports And Config Dataclasses
Imports, constants, `TrainConfig`, `Sample`, and small runtime helpers shared by the rest of the notebook.


In [ ]:
from __future__ import annotations

from contextlib import nullcontext
import json
import math
import os
import random
import shutil
import tempfile
import time
import zipfile
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace
from typing import Callable, Dict, List, Optional, Sequence, Tuple

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy.ndimage import label
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision.transforms import ElasticTransform, InterpolationMode
import torchvision.transforms.functional as TF
from tqdm import tqdm
from transformers import SegformerForSemanticSegmentation


ID2LABEL = {0: "crack"}
LABEL2ID = {"crack": 0}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
MASK_EXTS = [".png", ".bmp", ".jpg", ".jpeg", ".tif", ".tiff"]
MASK_HINTS = {"mask", "masks", "gt", "ground_truth", "label", "labels"}



@dataclass
class TrainConfig:
    output_dir: str
    data_root: Optional[str] = None
    drive_data_root: str = "/content/drive/MyDrive/KolektorSDD-boxes"
    local_data_root: str = "/content/KolektorSDD-boxes"
    copy_data_to_local: bool = True
    use_preprocessed_train: bool = True
    preprocess_cache_root: str = "/content/ksdd1_segformer_aug_cache"
    drive_preprocess_cache_zip: str = "/content/drive/MyDrive/ksdd1_segformer_aug_cache.zip"
    copy_preprocess_cache_from_drive: bool = True
    augmented_copies_per_sample: int = 24
    cache_copies_per_epoch: Optional[int] = 2
    force_rebuild_preprocess_cache: bool = False

    model_name: str = "nvidia/segformer-b0-finetuned-ade-512-512"
    seed: int = 42
    batch_size: int = 4
    eval_batch_size: int = 4
    image_size: Tuple[int, int] = (1408, 512)
    epochs: int = 50
    lr: float = 1e-4
    weight_decay: float = 1e-3
    grad_clip_norm: float = 1.0
    threshold: float = 0.95
    default_min_area: int = 1000
    require_cuda: bool = True
    preview_samples: int = 4
    num_workers: int = 0
    track_train_metrics: bool = True
    eval_train_metrics: bool = False
    run_threshold_sweep: bool = True
    threshold_sweep: str = "0.5,0.6,0.7,0.75,0.8,0.85,0.9,0.95,0.97,0.975,0.98,0.985,0.99"
    threshold_min_areas: Tuple[int, ...] = (0, 25, 50, 100, 200, 500, 1000, 1500, 2000)

    pos_weight: float = 10.0
    loss_mode: str = "focal_dice"
    focal_loss_weight: float = 0.4
    dice_loss_weight: float = 0.6
    dice_positive_only: bool = True
    focal_alpha: float = 0.75
    focal_gamma: float = 2.0

    selection_metric: str = "composite"
    scheduler_metric: str = "composite"
    image_fpr_penalty: float = 0.5
    threshold_pixel_fpr_lambda: float = 1.0
    early_stopping_patience: int = 5
    early_stopping_min_delta: float = 1e-4

    use_balanced_sampler: bool = True
    positive_sample_weight: float = 6.0

    elastic_prob: float = 0.3
    elastic_alpha: float = 25.0
    elastic_sigma: float = 5.0
    gaussian_noise_prob: float = 0.2
    gaussian_noise_std: float = 0.01

    lr_scheduler: str = "plateau"
    cosine_t_max: int = 30
    plateau_factor: float = 0.5
    plateau_patience: int = 2
    plateau_min_lr: float = 1e-6

    foreground_prior: Optional[float] = None
    prior_clamp: Tuple[float, float] = (1e-4, 1.0 - 1e-4)
    init_segmentation_bias: bool = True

    pad_divisor: int = 32

    strategy: str = "side_tuning"
    base_checkpoint_path: Optional[str] = None
    freeze_base_model: bool = True
    side_width: int = 32
    side_dropout: float = 0.10
    side_init_mix: float = 0.99
    side_lr_mult: float = 1.0
    mix_lr_mult: float = 5.0
    base_lr_mult: float = 0.1
    sync_timing: bool = False


@dataclass
class Sample:
    image_path: str
    mask_path: Optional[str]
    split: str
    sample_id: str
    label: int
    source_image: Optional[str] = None
    copy_idx: Optional[int] = None




METRIC_NAMES = ["dice", "dice_positive", "iou", "precision", "recall", "fpr", "image_fpr", "composite"]
TRACKED_HISTORY = ["loss"] + METRIC_NAMES
_TEMP_DATASETS: List[tempfile.TemporaryDirectory] = []


def mps_is_available() -> bool:
    return hasattr(torch.backends, "mps") and torch.backends.mps.is_available()


def get_best_device(cfg: TrainConfig) -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if mps_is_available():
        return torch.device("mps")
    if cfg.require_cuda:
        raise RuntimeError("CUDA is not available. In Colab, choose Runtime > Change runtime type > GPU.")
    return torch.device("cpu")


def amp_autocast(device: torch.device, use_amp: bool):
    if not use_amp:
        return nullcontext()
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast("cuda", dtype=torch.float16, enabled=use_amp)
    return torch.cuda.amp.autocast(dtype=torch.float16, enabled=use_amp)


def make_grad_scaler(use_amp: bool):
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        try:
            return torch.amp.GradScaler("cuda", enabled=use_amp)
        except TypeError:
            return torch.cuda.amp.GradScaler(enabled=use_amp)
    return torch.cuda.amp.GradScaler(enabled=use_amp)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if mps_is_available() and hasattr(torch, "mps") and hasattr(torch.mps, "manual_seed"):
        torch.mps.manual_seed(seed)


def move_to_device(tensor: torch.Tensor, device: torch.device) -> torch.Tensor:
    return tensor.to(device, non_blocking=(device.type == "cuda"))


def sync_device(device: torch.device, enabled: bool = True) -> None:
    if not enabled:
        return
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps" and hasattr(torch, "mps") and hasattr(torch.mps, "synchronize"):
        torch.mps.synchronize()


def load_weights(checkpoint_path: Path, device: torch.device):
    try:
        return torch.load(checkpoint_path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(checkpoint_path, map_location=device)


## Dataset Discovery
Locate the KolektorSDD kosXX folders, copy Drive data to Colab local disk when requested, and compute the foreground prior.


In [ ]:
def extract_zip_data_root(data_root: Path) -> Path:
    if data_root.is_file() and data_root.suffix.lower() == ".zip":
        tmp = tempfile.TemporaryDirectory(prefix="ksdd1_")
        _TEMP_DATASETS.append(tmp)
        out_dir = Path(tmp.name)
        print(f"Extracting dataset zip to temporary folder: {out_dir}")
        try:
            with zipfile.ZipFile(data_root, "r") as archive:
                archive.extractall(out_dir)
        except zipfile.BadZipFile as exc:
            size = data_root.stat().st_size if data_root.exists() else 0
            with open(data_root, "rb") as handle:
                first_bytes = handle.read(32)
            raise RuntimeError(
                "The file passed as data_root has a .zip extension, but Python cannot read it as a zip archive.\n"
                f"Path checked: {data_root}\n"
                f"File size: {size} bytes\n"
                f"First bytes: {first_bytes!r}\n"
                "Use the extracted KolektorSDD-boxes folder or a readable zip archive."
            ) from exc
        return out_dir
    return data_root


def find_child_dir_case_insensitive(parent: Path, name: str) -> Optional[Path]:
    if not parent.exists():
        return None
    wanted = name.lower()
    for child in parent.iterdir():
        if child.is_dir() and child.name.lower() == wanted:
            return child
    return None


def is_mask_path(path: Path) -> bool:
    lower = path.stem.lower()
    if lower.endswith("_gt") or any(token in lower for token in ("_label", "_mask")):
        return True
    if any(part.lower() in MASK_HINTS for part in path.parts):
        return True
    return False


def _has_train_test(root: Path) -> bool:
    return find_child_dir_case_insensitive(root, "train") is not None and find_child_dir_case_insensitive(root, "test") is not None


def _has_sdd1_groups(root: Path) -> bool:
    if not root.exists():
        return False
    for child in root.iterdir():
        if not child.is_dir():
            continue
        has_image = any(path.suffix.lower() in IMAGE_EXTS and not is_mask_path(path) for path in child.iterdir() if path.is_file())
        has_label = any(path.stem.lower().endswith("_label") for path in child.iterdir() if path.is_file())
        if has_image and has_label:
            return True
    return False


def resolve_data_root(data_root: Path) -> Path:
    root = extract_zip_data_root(Path(data_root))
    if not root.exists():
        raise FileNotFoundError(
            f"DATA_ROOT does not exist: {data_root}\n"
            "Set DATA_ROOT to KolektorSDD-boxes or a train/test dataset folder."
        )

    candidates = [root, root / "KolektorSDD-boxes", root / "KolektorSDD", root / "KolektorSDD2"]
    for candidate in candidates:
        if candidate.exists() and (_has_train_test(candidate) or _has_sdd1_groups(candidate)):
            return candidate

    for path in root.rglob("*"):
        if path.is_dir() and (_has_train_test(path) or _has_sdd1_groups(path)):
            return path

    raise FileNotFoundError(f"Could not find SDD1 kosXX folders or SDD2 train/test folders under: {root}")


def find_ksdd2_mask(image_path: Path) -> Optional[Path]:
    stems = [
        f"{image_path.stem}_GT",
        f"{image_path.stem}_gt",
        f"{image_path.stem}_label",
        f"{image_path.stem}_mask",
    ]
    suffixes = list(dict.fromkeys([image_path.suffix.lower()] + MASK_EXTS))
    for stem in stems:
        for suffix in suffixes:
            candidate = image_path.with_name(stem + suffix)
            if candidate.exists() and candidate.resolve() != image_path.resolve():
                return candidate
    return None


def mask_has_positive(mask_path: Optional[Path]) -> bool:
    if mask_path is None:
        return False
    mask = np.array(Image.open(mask_path).convert("L"))
    return bool((mask > 0).any())


def estimate_foreground_prior(samples: Sequence[Sample], cfg: TrainConfig) -> float:
    positive_pixels = 0
    total_pixels = 0

    for sample in tqdm(samples, desc="Estimating foreground prior", leave=False):
        if sample.mask_path is None:
            with Image.open(sample.image_path) as img:
                width, height = img.size
            total_pixels += width * height
            continue

        mask = np.array(Image.open(sample.mask_path).convert("L"))
        positive_pixels += int((mask > 0).sum())
        total_pixels += int(mask.size)

    prior = positive_pixels / max(total_pixels, 1)
    return float(np.clip(prior, *cfg.prior_clamp))


def collect_split_samples(root: Path, split_name: str) -> List[Sample]:
    split_dir = find_child_dir_case_insensitive(root, split_name) or (root / split_name)
    if not split_dir.is_dir():
        raise FileNotFoundError(f"Missing split folder: {split_dir}")

    samples: List[Sample] = []
    for image_path in sorted(split_dir.rglob("*")):
        if not image_path.is_file():
            continue
        if image_path.suffix.lower() not in IMAGE_EXTS:
            continue
        if "(copy)" in image_path.name.lower():
            continue
        if is_mask_path(image_path):
            continue

        mask_path = find_ksdd2_mask(image_path)
        sample_id = image_path.relative_to(split_dir).with_suffix("").as_posix()
        samples.append(
            Sample(
                image_path=str(image_path),
                mask_path=str(mask_path) if mask_path is not None else None,
                split=split_name,
                sample_id=sample_id,
                label=int(mask_has_positive(mask_path)),
            )
        )

    if not samples:
        raise RuntimeError(f"No input images found in {split_dir}")
    return samples


SDD1_VAL_FRACTION = 0.20


def collect_sdd1_samples(root: Path) -> List[Sample]:
    samples: List[Sample] = []
    for kos_folder in sorted(path for path in root.iterdir() if path.is_dir()):
        for image_path in sorted(kos_folder.glob("*.jpg")):
            if not image_path.is_file() or is_mask_path(image_path):
                continue
            mask_path = image_path.with_name(f"{image_path.stem}_label.bmp")
            if not mask_path.exists():
                mask_path = find_ksdd2_mask(image_path)
            samples.append(
                Sample(
                    image_path=str(image_path),
                    mask_path=str(mask_path) if mask_path is not None else None,
                    split="unassigned",
                    sample_id=f"{kos_folder.name}/{image_path.stem}",
                    label=int(mask_has_positive(mask_path)),
                )
            )
    if not samples:
        raise RuntimeError(f"No SDD1 kosXX/Part*.jpg samples found in {root}")
    return samples


def split_sdd1_samples(root: Path, val_fraction: float = SDD1_VAL_FRACTION) -> Tuple[List[Sample], List[Sample]]:
    samples = collect_sdd1_samples(root)
    positive = [sample for sample in samples if sample.label]
    negative = [sample for sample in samples if not sample.label]
    rng = random.Random(SEED)
    rng.shuffle(positive)
    rng.shuffle(negative)

    def split_class(items: List[Sample]) -> Tuple[List[Sample], List[Sample]]:
        if not items:
            return [], []
        n_val = max(1, int(round(len(items) * val_fraction)))
        n_val = min(n_val, max(1, len(items) - 1)) if len(items) > 1 else 1
        return items[n_val:], items[:n_val]

    pos_train, pos_val = split_class(positive)
    neg_train, neg_val = split_class(negative)
    if pos_train:
        neg_train = rng.sample(neg_train, min(len(neg_train), 2 * len(pos_train)))

    train_samples = [Sample(**{**sample.__dict__, "split": "train"}) for sample in pos_train + neg_train]
    val_samples = [Sample(**{**sample.__dict__, "split": "test"}) for sample in pos_val + neg_val]
    rng.shuffle(train_samples)
    rng.shuffle(val_samples)
    return train_samples, val_samples

def _split_counts(samples: Sequence[Sample]) -> Tuple[int, int]:
    positives = sum(sample.label for sample in samples)
    negatives = len(samples) - positives
    return positives, negatives


def load_official_split(data_root: str) -> Tuple[List[Sample], List[Sample]]:
    root = resolve_data_root(Path(data_root))
    if _has_train_test(root):
        train_samples = collect_split_samples(root, "train")
        val_samples = collect_split_samples(root, "test")
        split_name = "official train/test"
    else:
        train_samples, val_samples = split_sdd1_samples(root)
        split_name = "old KolektorSDD stratified sample split"

    train_pos, train_neg = _split_counts(train_samples)
    val_pos, val_neg = _split_counts(val_samples)

    print(f"Dataset root: {root}")
    print(f"Split: {split_name}")
    print(f"Train: {len(train_samples)} images ({train_pos} positive, {train_neg} negative)")
    print(f"Test/val: {len(val_samples)} images ({val_pos} positive, {val_neg} negative)")

    return train_samples, val_samples


def prepare_data_root(cfg: TrainConfig) -> Path:
    if cfg.data_root:
        return resolve_data_root(Path(cfg.data_root))

    drive_root = Path(cfg.drive_data_root)
    local_root = Path(cfg.local_data_root)
    if not cfg.copy_data_to_local:
        return resolve_data_root(drive_root)

    try:
        resolved_local = resolve_data_root(local_root)
        print(f"Using local dataset cache: {resolved_local}")
        return resolved_local
    except FileNotFoundError:
        pass

    source_root = resolve_data_root(drive_root)
    print(f"Copying dataset to Colab local disk: {source_root} -> {local_root}")
    shutil.copytree(source_root, local_root, dirs_exist_ok=True)
    return resolve_data_root(local_root)


## Preprocessed Augmentation Cache
Build or reuse a manifest-backed cache of augmented training samples. This is where expensive elastic/noise preprocessing is moved out of the epoch loop.


In [ ]:
def preprocess_cache_config(num_raw_samples: int, cfg: TrainConfig) -> Dict[str, float]:
    return {
        "num_raw_samples": int(num_raw_samples),
        "copies_per_sample": int(cfg.augmented_copies_per_sample),
        "seed": int(cfg.seed),
        "image_size": list(cfg.image_size),
        "elastic_prob": float(cfg.elastic_prob),
        "elastic_alpha": float(cfg.elastic_alpha),
        "elastic_sigma": float(cfg.elastic_sigma),
        "gaussian_noise_prob": float(cfg.gaussian_noise_prob),
        "gaussian_noise_std": float(cfg.gaussian_noise_std),
    }


def sample_to_manifest_row(sample: Sample) -> Dict[str, object]:
    return {
        "image_path": sample.image_path,
        "mask_path": sample.mask_path,
        "split": sample.split,
        "sample_id": sample.sample_id,
        "label": int(sample.label),
        "source_image": sample.source_image,
        "copy_idx": sample.copy_idx,
    }


def sample_from_manifest_row(row: Dict[str, object]) -> Sample:
    return Sample(
        image_path=str(row.get("image_path") or row.get("image")),
        mask_path=str(row.get("mask_path") or row.get("mask")) if (row.get("mask_path") or row.get("mask")) is not None else None,
        split=str(row.get("split", "train_preprocessed")),
        sample_id=str(row.get("sample_id", Path(str(row.get("image_path") or row.get("image"))).stem)),
        label=int(row.get("label", row.get("has_crack", 0))),
        source_image=row.get("source_image"),
        copy_idx=row.get("copy_idx"),
    )


def load_preprocess_manifest(cache_root: Path, expected_config: Dict[str, float]) -> Optional[List[Sample]]:
    manifest_path = cache_root / "manifest.json"
    if not manifest_path.exists():
        return None

    manifest = json.loads(manifest_path.read_text())
    if manifest.get("config") != expected_config:
        return None

    rows = manifest.get("samples", [])
    samples = [sample_from_manifest_row(row) for row in rows]
    for sample in samples:
        if not Path(sample.image_path).is_file():
            return None
        if sample.mask_path is not None and not Path(sample.mask_path).is_file():
            return None
    return samples


def restore_preprocess_cache_from_drive(cache_root: Path, expected_config: Dict[str, float], cfg: TrainConfig) -> Optional[List[Sample]]:
    if not cfg.copy_preprocess_cache_from_drive:
        return None

    drive_zip = Path(cfg.drive_preprocess_cache_zip)
    if not drive_zip.exists():
        print(f"Drive cache zip not found: {drive_zip}")
        return None

    cache_root = Path(cache_root)
    local_zip = PROJECT_ROOT / drive_zip.name
    print(f"Copying preprocessed cache zip from Drive: {drive_zip} -> {local_zip}")
    shutil.copy2(drive_zip, local_zip)

    if cache_root.exists():
        shutil.rmtree(cache_root)

    print(f"Unzipping preprocessed cache into {PROJECT_ROOT}")
    try:
        shutil.unpack_archive(str(local_zip), str(PROJECT_ROOT))
    finally:
        local_zip.unlink(missing_ok=True)

    cached_samples = load_preprocess_manifest(cache_root, expected_config)
    if cached_samples is None:
        print("Restored cache did not match the current notebook config; falling back to cache build.")
        if cache_root.exists():
            shutil.rmtree(cache_root)
        return None

    print(f"Restored preprocessed training cache: {cache_root} ({len(cached_samples)} samples)")
    return cached_samples


def save_cached_augmented_sample(sample: Sample, out_dir: Path, sample_idx: int, copy_idx: int, cfg: TrainConfig) -> Sample:
    img, mask = load_sample_pil(sample, cfg)
    img, mask = apply_training_augmentation(img, mask, cfg)

    img_t = maybe_add_gaussian_noise(TF.to_tensor(img), cfg)
    img_np = (img_t.permute(1, 2, 0).numpy() * 255.0).round().clip(0, 255).astype(np.uint8)
    mask_np = (np.array(mask) > 0).astype(np.uint8) * 255

    stem = f"{sample_idx:05d}_aug{copy_idx:02d}"
    image_path = out_dir / f"{stem}.png"
    mask_path = out_dir / f"{stem}_GT.png"

    Image.fromarray(img_np).save(image_path, compress_level=1)
    Image.fromarray(mask_np).save(mask_path, compress_level=1)

    return Sample(
        image_path=str(image_path),
        mask_path=str(mask_path),
        split="train_preprocessed",
        sample_id=stem,
        label=int(sample.label),
        source_image=sample.image_path,
        copy_idx=int(copy_idx),
    )


def build_preprocessed_train_cache(train_samples: Sequence[Sample], cfg: TrainConfig) -> List[Sample]:
    cache_root = Path(cfg.preprocess_cache_root)
    cache_config = preprocess_cache_config(len(train_samples), cfg)

    if cfg.force_rebuild_preprocess_cache and cache_root.exists():
        shutil.rmtree(cache_root)

    cached_samples = load_preprocess_manifest(cache_root, cache_config)
    if cached_samples is not None:
        print(f"Using preprocessed training cache: {cache_root} ({len(cached_samples)} samples)")
        return cached_samples

    cached_samples = restore_preprocess_cache_from_drive(cache_root, cache_config, cfg)
    if cached_samples is not None:
        return cached_samples

    out_dir = cache_root / "train"
    if cache_root.exists():
        shutil.rmtree(cache_root)
    out_dir.mkdir(parents=True, exist_ok=True)

    print(
        f"Preprocessing training augmentations to {cache_root} "
        f"({cfg.augmented_copies_per_sample} copy/copies per sample)"
    )
    start = time.time()
    py_rng_state = random.getstate()
    torch_rng_state = torch.random.get_rng_state()
    random.seed(cfg.seed)
    torch.manual_seed(cfg.seed)

    cached_samples: List[Sample] = []
    try:
        for sample_idx, sample in enumerate(tqdm(train_samples, desc="Preprocessing train cache")):
            for copy_idx in range(cfg.augmented_copies_per_sample):
                cached_samples.append(save_cached_augmented_sample(sample, out_dir, sample_idx, copy_idx, cfg))
    finally:
        random.setstate(py_rng_state)
        torch.random.set_rng_state(torch_rng_state)

    elapsed = time.time() - start
    manifest = {
        "config": cache_config,
        "samples": [sample_to_manifest_row(sample) for sample in cached_samples],
        "elapsed_seconds": elapsed,
        "cache_pool": {
            "copies_per_epoch": int(cfg.cache_copies_per_epoch),
            "description": "Runtime sampling hint only.",
        },
    }
    (cache_root / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print(f"Preprocessing complete in {elapsed / 60:.1f} min | Cached samples: {len(cached_samples)}")
    return cached_samples


def prepare_training_samples(train_samples: Sequence[Sample], cfg: TrainConfig) -> Tuple[List[Sample], bool]:
    if not cfg.use_preprocessed_train:
        print("Using online augmentation during training.")
        return list(train_samples), True

    cached_samples = build_preprocessed_train_cache(train_samples, cfg)
    return cached_samples, False


def make_epoch_subset(cached_samples: Sequence[Sample], copies_per_epoch: int, seed: int) -> List[Sample]:
    by_source: Dict[str, List[Sample]] = {}
    for sample in cached_samples:
        source_key = sample.source_image or sample.image_path
        by_source.setdefault(source_key, []).append(sample)

    rng = random.Random(seed)
    subset: List[Sample] = []
    for source_samples in by_source.values():
        k = min(int(copies_per_epoch), len(source_samples))
        subset.extend(rng.sample(source_samples, k))
    rng.shuffle(subset)
    return subset


## Dataset And Augmentations
Load PIL images/masks, resize to the old KolektorSDD training size, apply training augmentation, normalize tensors, and keep batch padding as a harmless safety net.


In [ ]:
def elastic_transform_pair(img: Image.Image, mask: Image.Image, cfg: TrainConfig) -> Tuple[Image.Image, Image.Image]:
    width, height = img.size
    displacement = ElasticTransform.get_params(
        alpha=[cfg.elastic_alpha, cfg.elastic_alpha],
        sigma=[cfg.elastic_sigma, cfg.elastic_sigma],
        size=[height, width],
    )
    img = TF.elastic_transform(
        img,
        displacement,
        interpolation=InterpolationMode.BILINEAR,
        fill=[0, 0, 0],
    )
    mask = TF.elastic_transform(
        mask,
        displacement,
        interpolation=InterpolationMode.NEAREST,
        fill=[0],
    )
    return img, mask


def load_sample_pil(sample: Sample, cfg: TrainConfig) -> Tuple[Image.Image, Image.Image]:
    image_path = Path(sample.image_path)
    mask_path = Path(sample.mask_path) if sample.mask_path is not None else None

    img = Image.open(image_path).convert("RGB")
    if mask_path is None:
        mask = Image.new("L", img.size, 0)
    else:
        mask = Image.open(mask_path).convert("L")
        if mask.size != img.size:
            raise ValueError(
                f"Image/mask size mismatch:\n"
                f"image={image_path}, size={img.size}\n"
                f"mask={mask_path}, size={mask.size}"
            )

    img = TF.resize(img, cfg.image_size, interpolation=InterpolationMode.BILINEAR)
    mask = TF.resize(mask, cfg.image_size, interpolation=InterpolationMode.NEAREST)
    return img, mask


def apply_training_augmentation(img: Image.Image, mask: Image.Image, cfg: TrainConfig) -> Tuple[Image.Image, Image.Image]:
    if random.random() < 0.5:
        img = TF.hflip(img)
        mask = TF.hflip(mask)
    if random.random() < 0.5:
        img = TF.vflip(img)
        mask = TF.vflip(mask)
    if random.random() < 0.3:
        img = TF.rotate(img, angle=180)
        mask = TF.rotate(mask, angle=180)

    if random.random() < cfg.elastic_prob:
        img, mask = elastic_transform_pair(img, mask, cfg)

    if random.random() < 0.4:
        img = TF.adjust_brightness(img, brightness_factor=random.uniform(0.9, 1.1))
        img = TF.adjust_contrast(img, contrast_factor=random.uniform(0.9, 1.1))

    return img, mask


def maybe_add_gaussian_noise(img_t: torch.Tensor, cfg: TrainConfig) -> torch.Tensor:
    if random.random() < cfg.gaussian_noise_prob:
        img_t = (img_t + torch.randn_like(img_t) * cfg.gaussian_noise_std).clamp(0.0, 1.0)
    return img_t


def normalize_image_tensor(img_t: torch.Tensor) -> torch.Tensor:
    return TF.normalize(
        img_t,
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )


class KSDD1Dataset(Dataset):
    def __init__(self, samples: Sequence[Sample], cfg: TrainConfig, augment: bool = True) -> None:
        self.samples = list(samples)
        self.cfg = cfg
        self.augment = augment

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        sample = self.samples[idx]
        img, mask = load_sample_pil(sample, self.cfg)

        if self.augment:
            img, mask = apply_training_augmentation(img, mask, self.cfg)

        img_t = TF.to_tensor(img)
        if self.augment:
            img_t = maybe_add_gaussian_noise(img_t, self.cfg)
        img_t = normalize_image_tensor(img_t)

        mask_np = np.array(mask)
        mask_t = torch.from_numpy((mask_np > 0).astype(np.float32))[None]
        return img_t, mask_t


def _ceil_to_multiple(x: int, divisor: int) -> int:
    if divisor is None or divisor <= 1:
        return x
    return int(np.ceil(x / divisor) * divisor)


def pad_collate(batch, pad_divisor: int):
    imgs, masks = zip(*batch)

    max_h = _ceil_to_multiple(max(img.shape[-2] for img in imgs), pad_divisor)
    max_w = _ceil_to_multiple(max(img.shape[-1] for img in imgs), pad_divisor)

    padded_imgs, padded_masks, valid_masks = [], [], []
    for img, mask in zip(imgs, masks):
        h, w = img.shape[-2:]
        pad_h = max_h - h
        pad_w = max_w - w

        padded_imgs.append(F.pad(img, (0, pad_w, 0, pad_h), value=0.0))
        padded_masks.append(F.pad(mask, (0, pad_w, 0, pad_h), value=0.0))

        valid = torch.zeros((1, max_h, max_w), dtype=torch.float32)
        valid[:, :h, :w] = 1.0
        valid_masks.append(valid)

    return torch.stack(padded_imgs), torch.stack(padded_masks), torch.stack(valid_masks)


## Model And Side-Tuning Strategies
Build a frozen SegFormer base, attach a trainable side U-Net branch, and configure the selected side-tuning strategy.


In [ ]:
def initialize_segmentation_bias(model: SegformerForSemanticSegmentation, prior_prob: float, cfg: TrainConfig):
    prior_prob = float(np.clip(prior_prob, *cfg.prior_clamp))
    bias_value = float(np.log(prior_prob / (1.0 - prior_prob)))

    classifier = getattr(model.decode_head, "classifier", None)
    if classifier is None or classifier.bias is None:
        print("Warning: no SegFormer decode_head.classifier bias found for prior initialization.")
        return model

    with torch.no_grad():
        classifier.bias.fill_(bias_value)
    print(f"Initialized SegFormer classifier bias with prior={prior_prob:.6f}, bias={bias_value:.4f}")
    return model


def initialize_conv_bias(conv: nn.Conv2d, prior_prob: float, cfg: TrainConfig) -> None:
    prior_prob = float(np.clip(prior_prob, *cfg.prior_clamp))
    bias_value = float(np.log(prior_prob / (1.0 - prior_prob)))
    if conv.bias is not None:
        nn.init.constant_(conv.bias, bias_value)


def logit_from_probability(prob: float) -> float:
    prob = float(np.clip(prob, 1e-4, 1.0 - 1e-4))
    return float(np.log(prob / (1.0 - prob)))


def set_requires_grad(module: nn.Module, requires_grad: bool) -> None:
    for parameter in module.parameters():
        parameter.requires_grad = requires_grad


def make_group_norm(channels: int, max_groups: int = 8) -> nn.GroupNorm:
    groups = min(max_groups, channels)
    while channels % groups != 0:
        groups -= 1
    return nn.GroupNorm(groups, channels)


class ConvGNAct(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            make_group_norm(out_channels),
            nn.SiLU(inplace=True),
            nn.Dropout2d(dropout) if dropout and dropout > 0 else nn.Identity(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            make_group_norm(out_channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class TinySideUNet(nn.Module):
    def __init__(self, cfg: TrainConfig, foreground_prior: Optional[float] = None) -> None:
        super().__init__()
        w = int(cfg.side_width)
        self.enc1 = ConvGNAct(3, w, dropout=cfg.side_dropout)
        self.enc2 = ConvGNAct(w, w * 2, dropout=cfg.side_dropout)
        self.enc3 = ConvGNAct(w * 2, w * 4, dropout=cfg.side_dropout)
        self.bottleneck = ConvGNAct(w * 4, w * 8, dropout=cfg.side_dropout)
        self.dec3 = ConvGNAct(w * 8 + w * 4, w * 4, dropout=cfg.side_dropout)
        self.dec2 = ConvGNAct(w * 4 + w * 2, w * 2, dropout=cfg.side_dropout)
        self.dec1 = ConvGNAct(w * 2 + w, w, dropout=cfg.side_dropout)
        self.head = nn.Conv2d(w, 1, kernel_size=1)
        if cfg.init_segmentation_bias and foreground_prior is not None:
            initialize_conv_bias(self.head, foreground_prior, cfg)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_shape = x.shape[-2:]
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, kernel_size=2))
        e3 = self.enc3(F.max_pool2d(e2, kernel_size=2))
        b = self.bottleneck(F.max_pool2d(e3, kernel_size=2))

        d3 = F.interpolate(b, size=e3.shape[-2:], mode="bilinear", align_corners=False)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = F.interpolate(d3, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = F.interpolate(d2, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        logits = self.head(d1)
        if logits.shape[-2:] != input_shape:
            logits = F.interpolate(logits, size=input_shape, mode="bilinear", align_corners=False)
        return logits


def build_base_segformer(cfg: TrainConfig, foreground_prior: Optional[float]) -> SegformerForSemanticSegmentation:
    model = SegformerForSemanticSegmentation.from_pretrained(
        cfg.model_name,
        num_labels=1,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )
    if cfg.init_segmentation_bias and foreground_prior is not None:
        model = initialize_segmentation_bias(model, foreground_prior, cfg)
    return model


def extract_state_dict(raw_state):
    if isinstance(raw_state, dict):
        for key in ("state_dict", "model_state_dict"):
            value = raw_state.get(key)
            if isinstance(value, dict):
                return value
    return raw_state


def strip_prefix(state: Dict[str, torch.Tensor], prefix: str) -> Dict[str, torch.Tensor]:
    return {
        key[len(prefix):] if key.startswith(prefix) else key: value
        for key, value in state.items()
    }


def select_compatible_state_dict(base_model: nn.Module, state: Dict[str, torch.Tensor]) -> Tuple[Dict[str, torch.Tensor], int, int]:
    base_state = base_model.state_dict()
    candidates = [state]
    for prefix in ("base_model.", "model.", "module.", "module.base_model."):
        if any(key.startswith(prefix) for key in state):
            candidates.append(strip_prefix(state, prefix))

    best = {}
    best_seen = 0
    for candidate in candidates:
        compatible = {
            key: value
            for key, value in candidate.items()
            if key in base_state and hasattr(value, "shape") and tuple(value.shape) == tuple(base_state[key].shape)
        }
        if len(compatible) > len(best):
            best = compatible
            best_seen = len(candidate)

    skipped = max(best_seen - len(best), 0)
    return best, len(best), skipped


def maybe_load_base_checkpoint(base_model: SegformerForSemanticSegmentation, cfg: TrainConfig, device: torch.device):
    if cfg.base_checkpoint_path is None:
        print(
            "BASE_CHECKPOINT_PATH is None. The frozen base branch uses the pretrained SegFormer checkpoint; "
            "SIDE_INIT_MIX should stay close to 1.0 for this ablation."
        )
        return base_model, False

    checkpoint_path = Path(cfg.base_checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"BASE_CHECKPOINT_PATH does not exist: {checkpoint_path}")

    raw_state = load_weights(checkpoint_path, device)
    state = extract_state_dict(raw_state)
    if not isinstance(state, dict):
        raise TypeError(f"Checkpoint does not contain a state dict: {checkpoint_path}")

    compatible_state, loaded_count, skipped_count = select_compatible_state_dict(base_model, state)
    base_tensor_count = len(base_model.state_dict())
    min_loaded = int(0.8 * base_tensor_count)
    if loaded_count < min_loaded:
        raise RuntimeError(
            f"Only {loaded_count}/{base_tensor_count} SegFormer base tensors matched {checkpoint_path}. "
            "Use a plain SegFormer checkpoint or a side-tuning checkpoint with base_model.* keys."
        )

    missing, unexpected = base_model.load_state_dict(compatible_state, strict=False)
    print(f"Loaded SegFormer base checkpoint: {checkpoint_path}")
    print(f"Base checkpoint tensors loaded: {loaded_count} | skipped: {skipped_count}")
    if missing:
        print(f"Base checkpoint missing keys after partial load: {len(missing)}")
    if unexpected:
        print(f"Base checkpoint unexpected keys after filtering: {len(unexpected)}")
    return base_model, True


class SideTuningSegformerModel(nn.Module):
    def __init__(
        self,
        base_model: SegformerForSemanticSegmentation,
        side_model: nn.Module,
        side_init_mix: float,
        freeze_base: bool = True,
    ) -> None:
        super().__init__()
        self.base_model = base_model
        self.side_model = side_model
        self.mix_logit = nn.Parameter(torch.tensor(logit_from_probability(side_init_mix), dtype=torch.float32))
        self.freeze_base = bool(freeze_base)
        self.set_base_trainable(not self.freeze_base)

    def set_base_trainable(self, trainable: bool):
        self.freeze_base = not bool(trainable)
        set_requires_grad(self.base_model, bool(trainable))
        if self.freeze_base:
            self.base_model.eval()
        return self

    def side_mix(self) -> torch.Tensor:
        return torch.sigmoid(self.mix_logit)

    def forward(self, pixel_values: torch.Tensor, **kwargs):
        if self.freeze_base:
            self.base_model.eval()
            with torch.no_grad():
                base_outputs = self.base_model(pixel_values=pixel_values, **kwargs)
        else:
            base_outputs = self.base_model(pixel_values=pixel_values, **kwargs)

        base_logits = base_outputs.logits
        side_logits = self.side_model(pixel_values)
        if base_logits.shape[-2:] != side_logits.shape[-2:]:
            base_logits = F.interpolate(
                base_logits,
                size=side_logits.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

        mix = self.side_mix().to(dtype=side_logits.dtype, device=side_logits.device)
        base_logits = base_logits.to(dtype=side_logits.dtype, device=side_logits.device)
        logits = (1.0 - mix) * base_logits + mix * side_logits
        return SimpleNamespace(logits=logits)


def iter_segformer_stages(base_model: SegformerForSemanticSegmentation):
    segformer = getattr(base_model, "segformer", None)
    stages = getattr(segformer, "stages", None)
    if stages is not None:
        for stage_idx, stage in enumerate(stages):
            yield f"segformer.stages.{stage_idx}", stage
        return

    encoder = getattr(segformer, "encoder", None)
    blocks = getattr(encoder, "block", None)
    if blocks is not None:
        for stage_idx, stage in enumerate(blocks):
            yield f"segformer.encoder.block.{stage_idx}", stage


def unfreeze_last_stage_and_decode_head(base_model: SegformerForSemanticSegmentation) -> str:
    stages = list(iter_segformer_stages(base_model))
    if not stages:
        raise RuntimeError("Could not find SegFormer stages for side_tuning_last_stage.")
    stage_name, stage = stages[-1]
    set_requires_grad(stage, True)
    if hasattr(base_model, "decode_head"):
        set_requires_grad(base_model.decode_head, True)
    return stage_name


def configure_side_tuning_strategy(model: SideTuningSegformerModel, cfg: TrainConfig) -> str:
    strategy = cfg.strategy.lower()
    if strategy not in AVAILABLE_STRATEGIES:
        raise ValueError(f"Unknown strategy={cfg.strategy!r}. Choose one of: {list(AVAILABLE_STRATEGIES)}")

    set_requires_grad(model, False)
    model.set_base_trainable(False)
    set_requires_grad(model.side_model, True)
    model.mix_logit.requires_grad = True

    if strategy == "side_tuning":
        return "frozen base + trainable side branch and mix"

    if strategy == "side_tuning_last_stage":
        stage_name = unfreeze_last_stage_and_decode_head(model.base_model)
        model.freeze_base = False
        return f"side branch + trainable {stage_name} and decode head"

    raise ValueError(f"Unknown side-tuning strategy: {strategy}")


def count_side_params(model: SideTuningSegformerModel) -> int:
    return sum(p.numel() for p in model.side_model.parameters())


def count_base_params(model: SideTuningSegformerModel) -> int:
    return sum(p.numel() for p in model.base_model.parameters())


def build_model(cfg: TrainConfig, foreground_prior: Optional[float], device: torch.device) -> SideTuningSegformerModel:
    base_model = build_base_segformer(cfg, foreground_prior)
    base_model, loaded_base = maybe_load_base_checkpoint(base_model, cfg, device)
    side_model = TinySideUNet(cfg, foreground_prior=foreground_prior)

    side_init_mix = cfg.side_init_mix
    if not loaded_base and side_init_mix < 0.95:
        print("No trained base checkpoint was loaded; overriding side mix to 0.99.")
        side_init_mix = 0.99

    model = SideTuningSegformerModel(
        base_model=base_model,
        side_model=side_model,
        side_init_mix=side_init_mix,
        freeze_base=cfg.freeze_base_model,
    )
    strategy_detail = configure_side_tuning_strategy(model, cfg)
    print(f"Side-tuning setup: {cfg.strategy} ({strategy_detail})")
    print(f"Initial side-branch mix alpha={model.side_mix().item():.4f}")
    return model.to(device)


## Losses And Metrics
Focal-Dice loss, small-component filtering, confusion counts, Dice/IoU/precision/recall, and the FPR-aware composite score.


In [ ]:
def dice_loss(logits: torch.Tensor, targets: torch.Tensor, valid: Optional[torch.Tensor], cfg: TrainConfig, eps=1e-6):
    if valid is None:
        valid = torch.ones_like(targets)

    probs = torch.sigmoid(logits) * valid
    targets = targets * valid

    if cfg.dice_positive_only:
        positive_samples = targets.sum(dim=(1, 2, 3)) > 0
        if not positive_samples.any():
            return logits.sum() * 0.0
        probs = probs[positive_samples]
        targets = targets[positive_samples]

    inter = (probs * targets).sum(dim=(2, 3))
    denom = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    return 1 - ((2 * inter + eps) / (denom + eps)).mean()


def bce_loss_masked(logits: torch.Tensor, targets: torch.Tensor, valid: Optional[torch.Tensor], cfg: TrainConfig):
    if valid is None:
        valid = torch.ones_like(targets)

    pos_weight = torch.tensor([cfg.pos_weight], device=logits.device)
    bce = nn.functional.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=pos_weight, reduction="none"
    )
    return (bce * valid).sum() / valid.sum().clamp_min(1.0)


def focal_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    valid: Optional[torch.Tensor],
    cfg: TrainConfig,
):
    if valid is None:
        valid = torch.ones_like(targets)

    bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    probs = torch.sigmoid(logits)
    pt = probs * targets + (1 - probs) * (1 - targets)
    alpha_t = cfg.focal_alpha * targets + (1 - cfg.focal_alpha) * (1 - targets)
    focal = alpha_t * (1 - pt).pow(cfg.focal_gamma) * bce * valid
    return focal.sum() / valid.sum().clamp_min(1.0)


def segmentation_loss(logits: torch.Tensor, targets: torch.Tensor, valid: Optional[torch.Tensor], cfg: TrainConfig):
    dice = dice_loss(logits, targets, valid, cfg)

    if cfg.loss_mode == "bce_dice":
        bce = bce_loss_masked(logits, targets, valid, cfg)
        return cfg.focal_loss_weight * bce + cfg.dice_loss_weight * dice
    if cfg.loss_mode == "focal_dice":
        focal = focal_loss(logits, targets, valid, cfg)
        return cfg.focal_loss_weight * focal + cfg.dice_loss_weight * dice

    raise ValueError(f"Unknown loss_mode: {cfg.loss_mode}")


def remove_small_components_np(mask: np.ndarray, min_area=0) -> np.ndarray:
    mask = mask.astype(bool, copy=False)
    if min_area is None or min_area <= 0:
        return mask

    structure = np.ones((3, 3), dtype=np.uint8)
    labeled, num_features = label(mask, structure=structure)
    if num_features == 0:
        return mask

    component_sizes = np.bincount(labeled.ravel())
    keep_component = component_sizes >= min_area
    keep_component[0] = False
    return keep_component[labeled]


def remove_small_components_batch(pred_bool: torch.Tensor, valid_bool: Optional[torch.Tensor] = None, min_area=0):
    if min_area is None or min_area <= 0:
        return pred_bool

    pred_cpu = pred_bool.detach().cpu().numpy().astype(bool)
    if valid_bool is not None:
        valid_cpu = valid_bool.detach().cpu().numpy().astype(bool)
    else:
        valid_cpu = np.ones_like(pred_cpu, dtype=bool)

    cleaned = np.zeros_like(pred_cpu, dtype=bool)
    for idx in range(pred_cpu.shape[0]):
        mask_2d = pred_cpu[idx, 0] & valid_cpu[idx, 0]
        cleaned[idx, 0] = remove_small_components_np(mask_2d, min_area=min_area)

    return torch.as_tensor(cleaned, device=pred_bool.device, dtype=torch.bool)


def selection_score(dice_positive: float, image_fpr: float, pixel_fpr: float, cfg: TrainConfig) -> float:
    return (
        dice_positive
        - cfg.image_fpr_penalty * image_fpr
        - cfg.threshold_pixel_fpr_lambda * pixel_fpr
    )


def confusion_counts(logits: torch.Tensor, targets: torch.Tensor, valid: Optional[torch.Tensor], cfg: TrainConfig, thr=None, min_area=0):
    if valid is None:
        valid = torch.ones_like(targets)
    if thr is None:
        thr = cfg.threshold

    valid_bool = valid > 0.5
    pred_bool = (torch.sigmoid(logits) > thr) & valid_bool
    pred_bool = remove_small_components_batch(pred_bool, valid_bool=valid_bool, min_area=min_area)
    target_bool = (targets > 0.5) & valid_bool

    per_tp = (pred_bool & target_bool).flatten(1).sum(dim=1).float()
    per_fp = (pred_bool & ~target_bool & valid_bool).flatten(1).sum(dim=1).float()
    per_fn = (~pred_bool & target_bool & valid_bool).flatten(1).sum(dim=1).float()

    tp = per_tp.sum().item()
    fp = per_fp.sum().item()
    fn = per_fn.sum().item()
    tn = (~pred_bool & ~target_bool & valid_bool).sum().item()

    gt_positive = target_bool.flatten(1).any(dim=1)
    pred_positive = pred_bool.flatten(1).any(dim=1)
    image_negatives = (~gt_positive).sum().item()
    image_fp = ((~gt_positive) & pred_positive).sum().item()

    dice_positive_sum = 0.0
    dice_positive_count = gt_positive.sum().item()
    if dice_positive_count > 0:
        positive_dice = (2 * per_tp[gt_positive] + 1e-6) / (
            2 * per_tp[gt_positive] + per_fp[gt_positive] + per_fn[gt_positive] + 1e-6
        )
        dice_positive_sum = positive_dice.sum().item()

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "dice_positive_sum": dice_positive_sum,
        "dice_positive_count": dice_positive_count,
        "image_fp": image_fp,
        "image_negatives": image_negatives,
    }


def empty_counts() -> Dict[str, float]:
    return {
        "tp": 0,
        "fp": 0,
        "fn": 0,
        "tn": 0,
        "dice_positive_sum": 0.0,
        "dice_positive_count": 0,
        "image_fp": 0,
        "image_negatives": 0,
    }


def add_counts(total: Dict[str, float], batch_counts: Dict[str, float]) -> None:
    for key in total:
        total[key] += batch_counts[key]


def metrics_from_counts(counts: Dict[str, float], cfg: TrainConfig, eps=1e-6) -> Dict[str, float]:
    tp = counts["tp"]
    fp = counts["fp"]
    fn = counts["fn"]
    tn = counts["tn"]

    dice_denom = 2 * tp + fp + fn
    iou_denom = tp + fp + fn
    precision_denom = tp + fp
    recall_denom = tp + fn
    fpr_denom = fp + tn
    dice_positive_count = counts["dice_positive_count"]
    image_fpr_denom = counts["image_negatives"]

    metrics = {
        "dice": (2 * tp + eps) / (dice_denom + eps) if dice_denom > 0 else 1.0,
        "dice_positive": counts["dice_positive_sum"] / (dice_positive_count + eps) if dice_positive_count > 0 else 0.0,
        "iou": (tp + eps) / (iou_denom + eps) if iou_denom > 0 else 1.0,
        "precision": tp / (precision_denom + eps),
        "recall": tp / (recall_denom + eps),
        "fpr": fp / (fpr_denom + eps) if fpr_denom > 0 else 0.0,
        "image_fpr": counts["image_fp"] / (image_fpr_denom + eps) if image_fpr_denom > 0 else 0.0,
    }
    metrics["composite"] = selection_score(
        metrics["dice_positive"],
        metrics["image_fpr"],
        metrics["fpr"],
        cfg,
    )
    return metrics


## Training Loop
Forward/backward timing, train/validation metrics, scheduler stepping, checkpointing, and early stopping.


In [ ]:
def forward_logits(model: SegformerForSemanticSegmentation, x: torch.Tensor, target_shape: Tuple[int, int]) -> torch.Tensor:
    outputs = model(pixel_values=x)
    logits = outputs.logits
    if logits.shape[-2:] == target_shape:
        return logits
    return F.interpolate(
        logits,
        size=target_shape,
        mode="bilinear",
        align_corners=False,
    )


def run_epoch(
    model: SegformerForSemanticSegmentation,
    loader: DataLoader,
    cfg: TrainConfig,
    device: torch.device,
    optimizer=None,
    scaler=None,
    use_amp: bool = False,
    train: bool = True,
    thr: Optional[float] = None,
    min_area: int = 0,
    compute_metrics: bool = True,
) -> Dict[str, float]:
    model.train() if train else model.eval()

    sync_device(device, cfg.sync_timing)
    epoch_start = time.perf_counter()
    model_seconds = 0.0
    metric_seconds = 0.0
    transfer_seconds = 0.0
    total_loss = 0.0
    total_samples = 0
    counts = empty_counts()

    for x, y, valid in tqdm(loader, leave=False):
        transfer_start = time.perf_counter()
        x = move_to_device(x, device)
        y = move_to_device(y, device)
        valid = move_to_device(valid, device)
        sync_device(device, cfg.sync_timing)
        transfer_seconds += time.perf_counter() - transfer_start

        sync_device(device, cfg.sync_timing)
        model_start = time.perf_counter()
        if train:
            optimizer.zero_grad(set_to_none=True)

        grad_context = nullcontext() if train else torch.no_grad()
        with grad_context:
            with amp_autocast(device, use_amp):
                logits = forward_logits(model, x, target_shape=y.shape[-2:])
                loss = segmentation_loss(logits, y, valid, cfg)

            if train:
                if scaler is None:
                    loss.backward()
                    if cfg.grad_clip_norm is not None and cfg.grad_clip_norm > 0:
                        nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.grad_clip_norm)
                    optimizer.step()
                else:
                    scaler.scale(loss).backward()
                    if cfg.grad_clip_norm is not None and cfg.grad_clip_norm > 0:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.grad_clip_norm)
                    scaler.step(optimizer)
                    scaler.update()
        sync_device(device, cfg.sync_timing)
        model_seconds += time.perf_counter() - model_start

        batch_size = x.size(0)
        total_loss += float(loss.detach().item()) * batch_size
        total_samples += batch_size

        if compute_metrics:
            sync_device(device, cfg.sync_timing)
            metric_start = time.perf_counter()
            batch_counts = confusion_counts(logits.detach(), y.detach(), valid.detach(), cfg, thr=thr, min_area=min_area)
            sync_device(device, cfg.sync_timing)
            metric_seconds += time.perf_counter() - metric_start
            add_counts(counts, batch_counts)

    if compute_metrics:
        metrics = metrics_from_counts(counts, cfg)
    else:
        metrics = {metric: None for metric in METRIC_NAMES}
    sync_device(device, cfg.sync_timing)
    elapsed = time.perf_counter() - epoch_start
    metrics["loss"] = total_loss / max(total_samples, 1)
    metrics["seconds"] = model_seconds
    metrics["model_seconds"] = model_seconds
    metrics["metric_seconds"] = metric_seconds
    metrics["transfer_seconds"] = transfer_seconds
    metrics["wall_seconds"] = elapsed
    metrics["other_seconds"] = max(0.0, elapsed - model_seconds - metric_seconds - transfer_seconds)
    metrics["samples_per_second"] = total_samples / max(model_seconds, 1e-9)
    metrics["wall_samples_per_second"] = total_samples / max(elapsed, 1e-9)
    return metrics


def scheduler_metric_mode(metric_name: str) -> str:
    return "min" if metric_name == "loss" else "max"


def make_scheduler(optimizer, cfg: TrainConfig):
    if cfg.lr_scheduler == "plateau":
        return ReduceLROnPlateau(
            optimizer,
            mode=scheduler_metric_mode(cfg.scheduler_metric),
            factor=cfg.plateau_factor,
            patience=cfg.plateau_patience,
            threshold=cfg.early_stopping_min_delta,
            threshold_mode="abs",
            min_lr=cfg.plateau_min_lr,
        )
    if cfg.lr_scheduler == "cosine":
        return CosineAnnealingLR(optimizer, T_max=cfg.cosine_t_max, eta_min=cfg.plateau_min_lr)
    raise ValueError(f"Unknown lr_scheduler: {cfg.lr_scheduler}")


def step_scheduler(scheduler, val_metrics: Dict[str, float], cfg: TrainConfig) -> None:
    if cfg.lr_scheduler == "plateau":
        if cfg.scheduler_metric not in val_metrics:
            raise KeyError(f"Unknown scheduler_metric={cfg.scheduler_metric!r}; available: {sorted(val_metrics)}")
        scheduler.step(val_metrics[cfg.scheduler_metric])
    else:
        scheduler.step()


def make_balanced_sampler(dataset: KSDD1Dataset, cfg: TrainConfig):
    labels = [bool(sample.label) for sample in dataset.samples]
    n_pos = sum(labels)
    n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None

    positive_weight = cfg.positive_sample_weight
    if positive_weight is None:
        positive_weight = n_neg / n_pos

    weights = [positive_weight if label else 1.0 for label in labels]
    generator = torch.Generator()
    generator.manual_seed(cfg.seed)
    return WeightedRandomSampler(
        weights=weights,
        num_samples=len(weights),
        replacement=True,
        generator=generator,
    )


def make_loader(dataset: KSDD1Dataset, cfg: TrainConfig, batch_size: int, shuffle: bool, balanced: bool = False):
    sampler = make_balanced_sampler(dataset, cfg) if balanced else None
    generator = torch.Generator()
    generator.manual_seed(cfg.seed)

    loader_kwargs = {
        "batch_size": batch_size,
        "shuffle": shuffle and sampler is None,
        "sampler": sampler,
        "generator": generator if (shuffle or sampler is not None) else None,
        "num_workers": cfg.num_workers,
        "pin_memory": False,
        "collate_fn": lambda batch: pad_collate(batch, cfg.pad_divisor),
    }
    if cfg.num_workers > 0:
        loader_kwargs["persistent_workers"] = False
        loader_kwargs["prefetch_factor"] = 2

    return DataLoader(dataset, **loader_kwargs)



def make_optimizer(model: nn.Module, cfg: TrainConfig):
    side_params = []
    mix_params = []
    base_params = []
    other_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name == "mix_logit":
            mix_params.append(param)
        elif name.startswith("side_model."):
            side_params.append(param)
        elif name.startswith("base_model."):
            base_params.append(param)
        else:
            other_params.append(param)

    param_groups = []
    if side_params:
        param_groups.append({"params": side_params, "lr": cfg.lr * cfg.side_lr_mult, "name": "side"})
    if mix_params:
        param_groups.append({"params": mix_params, "lr": cfg.lr * cfg.mix_lr_mult, "weight_decay": 0.0, "name": "mix"})
    if base_params:
        param_groups.append({"params": base_params, "lr": cfg.lr * cfg.base_lr_mult, "name": "base"})
    if other_params:
        param_groups.append({"params": other_params, "lr": cfg.lr, "name": "other"})
    if not param_groups:
        raise ValueError("No trainable parameters. Check the side-tuning strategy.")

    return torch.optim.AdamW(param_groups, weight_decay=cfg.weight_decay)


def format_lrs(optimizer) -> str:
    return ", ".join(
        f"{group.get('name', f'group{idx}')}={group['lr']:.2e}"
        for idx, group in enumerate(optimizer.param_groups)
    )


def train_model(
    model: SegformerForSemanticSegmentation,
    train_loader: Optional[DataLoader],
    val_loader: DataLoader,
    cfg: TrainConfig,
    device: torch.device,
    checkpoint_path: Path,
    use_amp: bool,
    train_loader_factory: Optional[Callable[[int], DataLoader]] = None,
    train_sample_count: Optional[int] = None,
) -> Tuple[Dict[str, List[float]], Dict[str, float]]:
    optimizer = make_optimizer(model, cfg)
    scheduler = make_scheduler(optimizer, cfg)
    scaler = make_grad_scaler(use_amp)

    history = {
        f"{phase}_{metric}": []
        for phase in ("train", "val")
        for metric in TRACKED_HISTORY
    }

    best_selection_score = -float("inf")
    best_metrics: Optional[Dict[str, float]] = None
    epochs_without_improvement = 0
    start = time.perf_counter()
    total_train_step_seconds = 0.0
    total_train_metric_seconds = 0.0
    total_train_transfer_seconds = 0.0
    total_val_wall_seconds = 0.0

    for epoch in range(1, cfg.epochs + 1):
        epoch_train_loader = train_loader_factory(epoch) if train_loader_factory is not None else train_loader
        if epoch_train_loader is None:
            raise ValueError("train_loader cannot be None when train_loader_factory is not provided.")
        train_metrics = run_epoch(
            model,
            epoch_train_loader,
            cfg,
            device,
            optimizer=optimizer,
            scaler=scaler,
            use_amp=use_amp,
            train=True,
            thr=cfg.threshold,
            min_area=cfg.default_min_area,
            compute_metrics=cfg.track_train_metrics,
        )
        val_metrics = run_epoch(
            model,
            val_loader,
            cfg,
            device,
            optimizer=None,
            scaler=None,
            use_amp=use_amp,
            train=False,
            thr=cfg.threshold,
            min_area=cfg.default_min_area,
            compute_metrics=True,
        )
        step_scheduler(scheduler, val_metrics, cfg)
        lr_text = format_lrs(optimizer)
        side_mix_text = f" | side_mix {model.side_mix().item():.3f}" if hasattr(model, "side_mix") else ""
        total_train_step_seconds += train_metrics["model_seconds"]
        total_train_metric_seconds += train_metrics["metric_seconds"]
        total_train_transfer_seconds += train_metrics["transfer_seconds"]
        total_val_wall_seconds += val_metrics["wall_seconds"]

        for metric in TRACKED_HISTORY:
            train_value = train_metrics[metric]
            history[f"train_{metric}"].append(float("nan") if train_value is None else train_value)
            history[f"val_{metric}"].append(val_metrics[metric])

        train_dice = train_metrics["dice"] if train_metrics["dice"] is not None else float("nan")
        train_dice_positive = train_metrics["dice_positive"] if train_metrics["dice_positive"] is not None else float("nan")
        train_iou = train_metrics["iou"] if train_metrics["iou"] is not None else float("nan")
        train_precision = train_metrics["precision"] if train_metrics["precision"] is not None else float("nan")
        train_recall = train_metrics["recall"] if train_metrics["recall"] is not None else float("nan")
        train_fpr = train_metrics["fpr"] if train_metrics["fpr"] is not None else float("nan")
        train_image_fpr = train_metrics["image_fpr"] if train_metrics["image_fpr"] is not None else float("nan")
        train_composite = train_metrics["composite"] if train_metrics["composite"] is not None else float("nan")

        current_score = val_metrics[cfg.selection_metric]
        improved = current_score > best_selection_score + cfg.early_stopping_min_delta

        saved = ""
        if improved:
            best_selection_score = current_score
            best_metrics = {"epoch": epoch, **val_metrics}
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
            saved = "  <- saved"
        else:
            epochs_without_improvement += 1

        print(
            f"[{epoch:3d}/{cfg.epochs}] "
            f"loss {train_metrics['loss']:.4f}/{val_metrics['loss']:.4f} | "
            f"dice {train_dice:.3f}/{val_metrics['dice']:.3f} | "
            f"dice_pos {train_dice_positive:.3f}/{val_metrics['dice_positive']:.3f} | "
            f"iou {train_iou:.3f}/{val_metrics['iou']:.3f} | "
            f"precision {train_precision:.3f}/{val_metrics['precision']:.3f} | "
            f"recall {train_recall:.3f}/{val_metrics['recall']:.3f} | "
            f"fpr {train_fpr:.5f}/{val_metrics['fpr']:.5f} | "
            f"image_fpr {train_image_fpr:.3f}/{val_metrics['image_fpr']:.3f} | "
            f"score {train_composite:.3f}/{val_metrics['composite']:.3f} | "
            f"lr {lr_text}"
            f"{side_mix_text} | "
            f"train_step {train_metrics['model_seconds']:.1f}s | "
            f"train_metrics {train_metrics['metric_seconds']:.1f}s | "
            f"val_wall {val_metrics['wall_seconds']:.1f}s"
            f"{saved}"
        )

        if epochs_without_improvement >= cfg.early_stopping_patience:
            print(
                f"Early stopping at epoch {epoch}: no {cfg.selection_metric} improvement "
                f"for {cfg.early_stopping_patience} epochs."
            )
            break

    elapsed = time.perf_counter() - start
    epochs_run = len(history["train_loss"])
    effective_train_samples = train_sample_count
    if effective_train_samples is None:
        if train_loader is None:
            raise ValueError("train_sample_count must be provided when train_loader_factory is used.")
        effective_train_samples = len(train_loader.dataset)
    seconds_per_epoch = elapsed / max(epochs_run, 1)
    train_samples_per_second = effective_train_samples / max(seconds_per_epoch, 1e-9)
    print(
        f"\nRun wall time: {elapsed / 60:.1f} min | "
        f"Actual optimizer training time: {total_train_step_seconds / 60:.1f} min | "
        f"Train metric time: {total_train_metric_seconds / 60:.1f} min | "
        f"Train transfer time: {total_train_transfer_seconds / 60:.1f} min | "
        f"Validation wall time: {total_val_wall_seconds / 60:.1f} min"
    )
    print(
        f"Runtime: {elapsed:.1f}s total | {seconds_per_epoch:.1f}s/epoch | "
        f"{train_samples_per_second:.1f} train samples/s"
    )
    print(f"Best validation {cfg.selection_metric}: {best_selection_score:.4f}")

    if best_metrics is None:
        raise RuntimeError("Training finished without a best checkpoint.")
    best_metrics["training_seconds"] = elapsed
    best_metrics["run_wall_seconds"] = elapsed
    best_metrics["total_train_step_seconds"] = total_train_step_seconds
    best_metrics["total_train_metric_seconds"] = total_train_metric_seconds
    best_metrics["total_train_transfer_seconds"] = total_train_transfer_seconds
    best_metrics["total_val_wall_seconds"] = total_val_wall_seconds
    best_metrics["seconds_per_epoch"] = seconds_per_epoch
    best_metrics["epochs_run"] = epochs_run
    best_metrics["train_samples_per_epoch"] = effective_train_samples
    best_metrics["train_samples_per_second"] = train_samples_per_second
    return history, best_metrics


## Plots And Prediction Preview
Learning-curve plotting and visual prediction overlays for quick qualitative inspection.


In [ ]:
def plot_history(history: Dict[str, List[float]], out_path: Path, title_prefix: str = "SegFormer full_ft") -> None:
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(f"KSDD1 {title_prefix} learning curves (focal/dice)", fontsize=14)

    axes[0, 0].plot(epochs, history["train_loss"], label="Train")
    axes[0, 0].plot(epochs, history["val_loss"], label="Val")
    axes[0, 0].set_title("Loss")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(epochs, history["train_dice"], label="Train Dice")
    axes[0, 1].plot(epochs, history["val_dice"], label="Val Dice")
    axes[0, 1].plot(epochs, history["train_dice_positive"], label="Train Dice positive", linestyle="-.")
    axes[0, 1].plot(epochs, history["val_dice_positive"], label="Val Dice positive", linestyle="-.")
    axes[0, 1].plot(epochs, history["train_iou"], label="Train IoU", linestyle="--")
    axes[0, 1].plot(epochs, history["val_iou"], label="Val IoU", linestyle="--")
    axes[0, 1].set_title("Overlap")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Score")
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)

    axes[1, 0].plot(epochs, history["train_precision"], label="Train Precision")
    axes[1, 0].plot(epochs, history["val_precision"], label="Val Precision")
    axes[1, 0].plot(epochs, history["train_recall"], label="Train Recall", linestyle="--")
    axes[1, 0].plot(epochs, history["val_recall"], label="Val Recall", linestyle="--")
    axes[1, 0].set_title("Precision and Recall")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Score")
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)

    axes[1, 1].plot(epochs, history["train_fpr"], label="Train pixel FPR")
    axes[1, 1].plot(epochs, history["val_fpr"], label="Val pixel FPR")
    axes[1, 1].plot(epochs, history["train_image_fpr"], label="Train image FPR", linestyle="--")
    axes[1, 1].plot(epochs, history["val_image_fpr"], label="Val image FPR", linestyle="--")
    axes[1, 1].set_title("False Positive Rates")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("FPR")
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: {out_path}")


def print_result(result: Dict[str, float]) -> None:
    print(f"\nBest result for strategy: {result['strategy']}")
    print("-" * 50)
    print(f"checkpoint: {result['checkpoint_path']}")
    print(f"best_epoch: {result['best_epoch']}")
    print(f"trainable_params: {result['trainable_params']:,} / {result['total_params']:,}")
    print(f"foreground_prior: {result['foreground_prior']:.6f}")

    for metric in TRACKED_HISTORY:
        value = result[f"best_val_{metric}"]
        print(f"best_val_{metric}: {value:.4f}")


def visualise_predictions(
    model: SegformerForSemanticSegmentation,
    val_loader: DataLoader,
    checkpoint_path: Path,
    cfg: TrainConfig,
    device: torch.device,
    out_path: Path,
    n: int = 4,
    thr: Optional[float] = None,
    min_area: int = 0,
) -> None:
    if thr is None:
        thr = cfg.threshold
    model.load_state_dict(load_weights(checkpoint_path, device))
    model.eval()

    x_batch, y_batch, valid_batch = next(iter(val_loader))
    x_dev = move_to_device(x_batch, device)

    with torch.no_grad():
        logits = forward_logits(model, x_dev, target_shape=y_batch.shape[-2:])
    probs = torch.sigmoid(logits).cpu().numpy()

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    imgs = x_batch.numpy().transpose(0, 2, 3, 1)
    imgs = (imgs * std + mean).clip(0, 1)

    masks_gt = y_batch.numpy()[:, 0]
    masks_pred = probs[:, 0]
    valid = valid_batch.numpy()[:, 0]

    n = min(n, imgs.shape[0])
    fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
    if n == 1:
        axes = axes[np.newaxis]

    pred_title = f"Prediction (thr={thr:.2f}"
    if min_area > 0:
        pred_title += f", min_area={min_area}"
    pred_title += ")"
    col_titles = ["Image", "Ground truth", pred_title, "Overlay"]
    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=12)

    for i in range(n):
        ys, xs = np.where(valid[i] > 0)
        h = ys.max() + 1 if len(ys) else imgs.shape[1]
        w = xs.max() + 1 if len(xs) else imgs.shape[2]

        img_i = imgs[i, :h, :w]
        gt_i = masks_gt[i, :h, :w] > 0
        pred_i = masks_pred[i, :h, :w] > thr
        pred_i = remove_small_components_np(pred_i, min_area=min_area)

        axes[i, 0].imshow(img_i)
        axes[i, 0].axis("off")
        axes[i, 1].imshow(gt_i, cmap="gray", vmin=0, vmax=1)
        axes[i, 1].axis("off")
        axes[i, 2].imshow(pred_i, cmap="gray", vmin=0, vmax=1)
        axes[i, 2].axis("off")

        overlay = img_i.copy()
        overlay[pred_i & gt_i] = [0.0, 1.0, 0.0]
        overlay[pred_i & ~gt_i] = [1.0, 0.0, 0.0]
        overlay[~pred_i & gt_i] = [0.0, 0.0, 1.0]
        axes[i, 3].imshow(overlay)
        axes[i, 3].axis("off")

    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: {out_path}")


## Threshold Sweep
Cache validation probabilities once, then sweep probability thresholds and minimum connected-component areas cheaply.


In [ ]:
def parse_thresholds(value: str) -> List[float]:
    thresholds = []
    for item in value.split(","):
        item = item.strip()
        if item:
            thresholds.append(float(item))
    return thresholds


def threshold_score(row: Dict[str, float], cfg: TrainConfig) -> float:
    return selection_score(
        row["val_dice_positive"],
        row["val_image_fpr"],
        row["val_fpr"],
        cfg,
    )


def confusion_counts_from_arrays(prob_batch, target_batch, valid_batch, cfg: TrainConfig, thr=None, min_area=0):
    if thr is None:
        thr = cfg.threshold
    counts = empty_counts()

    for prob, target, valid in zip(prob_batch, target_batch, valid_batch):
        valid_bool = valid > 0.5
        pred_bool = (prob > thr) & valid_bool
        pred_bool = remove_small_components_np(pred_bool, min_area=min_area)
        target_bool = (target > 0.5) & valid_bool

        tp = int((pred_bool & target_bool).sum())
        fp = int((pred_bool & ~target_bool & valid_bool).sum())
        fn = int((~pred_bool & target_bool & valid_bool).sum())
        tn = int((~pred_bool & ~target_bool & valid_bool).sum())

        counts["tp"] += tp
        counts["fp"] += fp
        counts["fn"] += fn
        counts["tn"] += tn

        gt_positive = bool(target_bool.any())
        pred_positive = bool(pred_bool.any())
        if not gt_positive:
            counts["image_negatives"] += 1
            counts["image_fp"] += int(pred_positive)
        else:
            counts["dice_positive_count"] += 1
            counts["dice_positive_sum"] += (2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6)

    return counts


def evaluate_thresholds(
    model: SegformerForSemanticSegmentation,
    val_loader: DataLoader,
    checkpoint_path: Path,
    thresholds: Sequence[float],
    cfg: TrainConfig,
    device: torch.device,
    min_areas: Optional[Sequence[int]] = None,
):
    model.load_state_dict(load_weights(checkpoint_path, device))
    model.eval()

    if min_areas is None:
        min_areas = [0]

    cached_batches = []
    with torch.no_grad():
        for x_batch, y_batch, valid_batch in tqdm(val_loader, desc="Caching validation predictions", leave=False):
            x_dev = move_to_device(x_batch, device)
            logits = forward_logits(model, x_dev, target_shape=y_batch.shape[-2:])
            probs = torch.sigmoid(logits).cpu().numpy()[:, 0]
            cached_batches.append((
                probs,
                y_batch.numpy()[:, 0],
                valid_batch.numpy()[:, 0],
            ))

    results = []
    for min_area in min_areas:
        for thr in thresholds:
            counts = empty_counts()
            for probs, targets, valid in cached_batches:
                add_counts(
                    counts,
                    confusion_counts_from_arrays(
                        probs,
                        targets,
                        valid,
                        cfg,
                        thr=thr,
                        min_area=min_area,
                    ),
                )

            metrics = metrics_from_counts(counts, cfg)
            row = {"thr": thr, "min_area": min_area}
            row.update({f"val_{metric}": metrics[metric] for metric in METRIC_NAMES})
            row["selection_score"] = threshold_score(row, cfg)
            results.append(row)

    print("\nThreshold and min-area sweep:")
    for row in results:
        print(
            f"thr={row['thr']:.2f} | "
            f"min_area={row['min_area']:>3} | "
            f"score={row['selection_score']:.4f} | "
            f"dice={row['val_dice']:.4f} | "
            f"dice_pos={row['val_dice_positive']:.4f} | "
            f"iou={row['val_iou']:.4f} | "
            f"precision={row['val_precision']:.4f} | "
            f"recall={row['val_recall']:.4f} | "
            f"pixel_fpr={row['val_fpr']:.6f} | "
            f"image_fpr={row['val_image_fpr']:.4f}"
        )

    best = max(
        results,
        key=lambda row: (
            row["selection_score"],
            row["val_dice_positive"],
            -row["val_image_fpr"],
            -row["val_fpr"],
        ),
    )
    print(
        "\nBest threshold/post-processing by "
        "dice_positive - image_fpr penalty: "
        f"thr={best['thr']:.2f}, min_area={best['min_area']}"
    )
    return results, best


## Experiment Orchestration
This ties together data loading, model construction, training, threshold sweep, saved artifacts, and positive-sample previews.


In [ ]:
def side_mix_from_checkpoint(checkpoint_path: Path, device: torch.device) -> Optional[float]:
    state = load_weights(checkpoint_path, device)
    if isinstance(state, dict) and "state_dict" in state and isinstance(state["state_dict"], dict):
        state = state["state_dict"]
    mix_logit = state.get("mix_logit") if isinstance(state, dict) else None
    if mix_logit is None:
        return None
    return torch.sigmoid(mix_logit.float()).item()


def run_experiment(cfg: TrainConfig):
    set_seed(cfg.seed)
    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    device = get_best_device(cfg)
    use_amp = device.type == "cuda"
    print(f"Using device: {device}")
    if device.type == "cuda":
        print(f"CUDA GPU: {torch.cuda.get_device_name(0)}")
        print(f"Mixed precision AMP: {'enabled' if use_amp else 'disabled'}")
    elif device.type == "mps":
        print("MPS fallback enabled. Set require_cuda=False if you intentionally want to run outside Colab CUDA.")
    else:
        print("CUDA is not available. In Colab, choose Runtime > Change runtime type > GPU.")

    data_root = prepare_data_root(cfg)
    raw_train_samples, val_samples = load_official_split(str(data_root))
    train_samples, train_augment_online = prepare_training_samples(raw_train_samples, cfg)
    foreground_prior = cfg.foreground_prior
    if foreground_prior is None:
        foreground_prior = estimate_foreground_prior(raw_train_samples, cfg)
    else:
        foreground_prior = float(np.clip(foreground_prior, *cfg.prior_clamp))

    use_cache_pool = (
        cfg.use_preprocessed_train
        and not train_augment_online
        and cfg.cache_copies_per_epoch is not None
        and cfg.cache_copies_per_epoch < cfg.augmented_copies_per_sample
    )

    train_loader_factory = None
    train_sample_count = None
    if use_cache_pool:
        def train_loader_factory(epoch: int) -> DataLoader:
            epoch_samples = make_epoch_subset(
                train_samples,
                cfg.cache_copies_per_epoch,
                seed=cfg.seed + epoch,
            )
            epoch_ds = KSDD1Dataset(epoch_samples, cfg, augment=False)
            return make_loader(epoch_ds, cfg, cfg.batch_size, shuffle=True, balanced=cfg.use_balanced_sampler)

        train_loader = None
        epoch_summary_samples = make_epoch_subset(train_samples, cfg.cache_copies_per_epoch, seed=cfg.seed)
        train_sample_count = len(epoch_summary_samples)
    else:
        train_ds = KSDD1Dataset(train_samples, cfg, augment=train_augment_online)
        train_loader = make_loader(train_ds, cfg, cfg.batch_size, shuffle=True, balanced=cfg.use_balanced_sampler)
        epoch_summary_samples = train_samples
        train_sample_count = len(train_ds)

    train_eval_ds = KSDD1Dataset(raw_train_samples, cfg, augment=False)
    val_ds = KSDD1Dataset(val_samples, cfg, augment=False)

    train_eval_loader = make_loader(train_eval_ds, cfg, cfg.eval_batch_size, shuffle=False)
    val_loader = make_loader(val_ds, cfg, cfg.eval_batch_size, shuffle=False)

    strategy_name = cfg.strategy.lower()
    model = build_model(cfg, foreground_prior, device)
    initial_side_mix = float(model.side_mix().detach().cpu().item())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    side_params = count_side_params(model)
    base_params = count_base_params(model)
    checkpoint_path = output_dir / f"best_ksdd1_segformer_{strategy_name}.pth"

    print(f"\nStrategy: {strategy_name}")
    print(
        f"Focal-Dice loss: focal_weight={cfg.focal_loss_weight}, dice_weight={cfg.dice_loss_weight}, "
        f"alpha={cfg.focal_alpha}, gamma={cfg.focal_gamma}, foreground_prior={foreground_prior:.6f}"
    )
    print(
        f"All-sample augmentation: elastic_p={cfg.elastic_prob}, "
        f"noise_p={cfg.gaussian_noise_prob}, noise_std={cfg.gaussian_noise_std}"
    )
    print(
        "Training preprocessing: "
        f"{'precomputed cache' if cfg.use_preprocessed_train else 'online per epoch'} "
        f"| cached samples={len(train_samples)} | epoch samples={train_sample_count}"
    )
    if use_cache_pool:
        print(
            f"Cache pool sampling: drawing {cfg.cache_copies_per_epoch} "
            f"of {cfg.augmented_copies_per_sample} cached copies per source image each epoch"
        )
        print("Cache pool config is controlled by the notebook settings above; keep them in sync with the local cache script output.")
    print(
        "Epoch metrics: "
        f"train={'full' if cfg.track_train_metrics else 'loss-only fast path'}, validation=full"
    )
    print(
        f"Regularization: weight_decay={cfg.weight_decay}, grad_clip_norm={cfg.grad_clip_norm}"
    )
    print(f"Mixed precision AMP: {'enabled' if use_amp else 'disabled'}")
    if cfg.lr_scheduler == "plateau":
        print(
            f"LR scheduler: ReduceLROnPlateau on val_{cfg.scheduler_metric} "
            f"(mode={scheduler_metric_mode(cfg.scheduler_metric)}, factor={cfg.plateau_factor}, "
            f"patience={cfg.plateau_patience}, min_lr={cfg.plateau_min_lr})"
        )
    else:
        print(f"LR scheduler: CosineAnnealingLR (T_max={cfg.cosine_t_max}, min_lr={cfg.plateau_min_lr})")
    if cfg.use_balanced_sampler:
        n_pos = sum(sample.label for sample in epoch_summary_samples)
        n_neg = len(epoch_summary_samples) - n_pos
        sampler_weight = cfg.positive_sample_weight if cfg.positive_sample_weight is not None else n_neg / max(n_pos, 1)
        print(f"Balanced sampler: enabled, positive sample weight={sampler_weight:.2f}")
    else:
        print("Balanced sampler: disabled")
    print(
        f"FPR-aware score: dice_positive - {cfg.image_fpr_penalty} * image_fpr "
        f"- {cfg.threshold_pixel_fpr_lambda} * pixel_fpr"
    )
    print(f"Validation metric post-processing: threshold={cfg.threshold:.2f}, min_area={cfg.default_min_area}")
    print(f"Model selection / early stopping metric: val_{cfg.selection_metric}")
    print(
        f"Side tuning: side_width={cfg.side_width}, dropout={cfg.side_dropout}, "
        f"init_side_mix={initial_side_mix:.4f}, base_checkpoint={cfg.base_checkpoint_path}"
    )
    print(
        f"Parameter groups: side_lr={cfg.lr * cfg.side_lr_mult:.2e}, "
        f"mix_lr={cfg.lr * cfg.mix_lr_mult:.2e}, base_lr={cfg.lr * cfg.base_lr_mult:.2e}"
    )
    print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")
    print(f"Side branch parameters: {side_params:,} | Base branch parameters: {base_params:,}")
    print(f"Checkpoint: {checkpoint_path}")

    history, best_metrics = train_model(
        model,
        train_loader,
        val_loader,
        cfg,
        device,
        checkpoint_path=checkpoint_path,
        use_amp=use_amp,
        train_loader_factory=train_loader_factory,
        train_sample_count=train_sample_count,
    )

    if cfg.eval_train_metrics:
        best_train_metrics = run_epoch(
            model,
            train_eval_loader,
            cfg,
            device,
            optimizer=None,
            scaler=None,
            use_amp=use_amp,
            train=False,
            thr=cfg.threshold,
            min_area=cfg.default_min_area,
        )
        print(f"Final train-eval dice: {best_train_metrics['dice']:.4f}")

    result = {
        "strategy": strategy_name,
        "checkpoint_path": str(checkpoint_path),
        "best_epoch": best_metrics["epoch"],
        "trainable_params": trainable_params,
        "total_params": total_params,
        "foreground_prior": foreground_prior,
        "base_checkpoint_path": cfg.base_checkpoint_path,
        "side_width": cfg.side_width,
        "side_dropout": cfg.side_dropout,
        "initial_side_mix": initial_side_mix,
        "best_checkpoint_side_mix": side_mix_from_checkpoint(checkpoint_path, device),
        "side_params": side_params,
        "base_params": base_params,
        "cache_copies_per_epoch": cfg.cache_copies_per_epoch if use_cache_pool else None,
    }
    for metric in TRACKED_HISTORY:
        result[f"best_val_{metric}"] = best_metrics[metric]
    for timing_key in (
        "training_seconds",
        "run_wall_seconds",
        "seconds_per_epoch",
        "epochs_run",
        "train_samples_per_epoch",
        "train_samples_per_second",
        "total_train_step_seconds",
        "total_train_metric_seconds",
        "total_train_transfer_seconds",
        "total_val_wall_seconds",
    ):
        result[timing_key] = best_metrics[timing_key]

    history_path = output_dir / f"comparison_history_{strategy_name}.json"
    result_path = output_dir / f"best_result_{strategy_name}.json"
    threshold_path = output_dir / f"threshold_sweep_ksdd1_{strategy_name}.json"
    curves_path = output_dir / f"learning_curves_ksdd1_{strategy_name}.png"
    predictions_path = output_dir / f"predictions_ksdd1_{strategy_name}.png"
    result.update(
        {
            "history_path": str(history_path),
            "result_path": str(result_path),
            "threshold_sweep_path": str(threshold_path),
            "learning_curves_path": str(curves_path),
            "predictions_path": str(predictions_path),
        }
    )

    with open(history_path, "w", encoding="utf-8") as handle:
        json.dump(history, handle, indent=2)
    with open(result_path, "w", encoding="utf-8") as handle:
        json.dump(result, handle, indent=2)
    with open(output_dir / "split_summary.json", "w", encoding="utf-8") as handle:
        json.dump(
            {
                "source": "KSDD1 old KolektorSDD stratified split",
                "data_root": str(data_root),
                "train_total": len(raw_train_samples),
                "train_positive": sum(sample.label for sample in raw_train_samples),
                "train_negative": len(raw_train_samples) - sum(sample.label for sample in raw_train_samples),
                "epoch_train_total": len(epoch_summary_samples),
                "epoch_train_positive": sum(sample.label for sample in epoch_summary_samples),
                "epoch_train_negative": len(epoch_summary_samples) - sum(sample.label for sample in epoch_summary_samples),
                "test_total": len(val_samples),
                "test_positive": sum(sample.label for sample in val_samples),
                "test_negative": len(val_samples) - sum(sample.label for sample in val_samples),
                "preprocessing": "original size, ImageNet normalization, dynamic batch padding",
                "training_preprocessing": "precomputed cache" if cfg.use_preprocessed_train else "online per epoch",
                "preprocess_cache_root": str(cfg.preprocess_cache_root),
                "augmented_copies_per_sample": cfg.augmented_copies_per_sample,
                "cache_copies_per_epoch": cfg.cache_copies_per_epoch if use_cache_pool else None,
                "track_train_metrics": cfg.track_train_metrics,
                "eval_train_metrics": cfg.eval_train_metrics,
                "strategy": strategy_name,
                "base_checkpoint_path": cfg.base_checkpoint_path,
                "side_width": cfg.side_width,
                "side_dropout": cfg.side_dropout,
                "initial_side_mix": initial_side_mix,
                "side_lr_mult": cfg.side_lr_mult,
                "mix_lr_mult": cfg.mix_lr_mult,
                "base_lr_mult": cfg.base_lr_mult,
                "selection_metric": cfg.selection_metric,
                "scheduler_metric": cfg.scheduler_metric,
                "threshold": cfg.threshold,
                "default_min_area": cfg.default_min_area,
                "run_threshold_sweep": cfg.run_threshold_sweep,
                "threshold_sweep": cfg.threshold_sweep,
                "threshold_min_areas": list(cfg.threshold_min_areas),
                "amp": use_amp,
                "pad_divisor": cfg.pad_divisor,
            },
            handle,
            indent=2,
        )

    print_result(result)
    plot_history(history, curves_path, title_prefix=f"SegFormer {strategy_name}")

    if cfg.run_threshold_sweep:
        thresholds = parse_thresholds(cfg.threshold_sweep)
        threshold_results, best_thr = evaluate_thresholds(
            model,
            val_loader,
            checkpoint_path=checkpoint_path,
            thresholds=thresholds,
            cfg=cfg,
            device=device,
            min_areas=cfg.threshold_min_areas,
        )
    else:
        print(
            "Skipping threshold/min-area sweep; "
            f"using thr={cfg.threshold:.2f}, min_area={cfg.default_min_area} for previews."
        )
        threshold_results = []
        best_thr = {"thr": cfg.threshold, "min_area": cfg.default_min_area}
    with open(threshold_path, "w", encoding="utf-8") as handle:
        json.dump({"rows": threshold_results, "best": best_thr}, handle, indent=2)

    visualise_predictions(
        model,
        val_loader,
        checkpoint_path=checkpoint_path,
        cfg=cfg,
        device=device,
        out_path=predictions_path,
        n=cfg.preview_samples,
        thr=best_thr["thr"],
        min_area=best_thr.get("min_area", 0),
    )

    print(f"Outputs saved to: {output_dir}")
    return history, result, model, val_loader, best_thr


def visualise_positive_predictions(
    cfg: TrainConfig,
    result: Dict[str, float],
    best_thr: Dict[str, float],
    split: str = "test",
    n: int = 4,
) -> None:
    device = get_best_device(cfg)
    data_root = prepare_data_root(cfg)
    train_samples, val_samples = load_official_split(str(data_root))
    samples = val_samples if split == "test" else train_samples
    positive_samples = [sample for sample in samples if sample.label]
    if not positive_samples:
        raise RuntimeError(f"No defective images found in split={split!r}")

    rng = random.Random(cfg.seed)
    selected_samples = positive_samples.copy()
    rng.shuffle(selected_samples)
    selected_samples = selected_samples[:n]

    print(f"Showing {len(selected_samples)} defective images from {split} split")
    for sample in selected_samples:
        print(Path(sample.image_path).name)

    selected_loader = make_loader(
        KSDD1Dataset(selected_samples, cfg, augment=False),
        cfg,
        batch_size=len(selected_samples),
        shuffle=False,
    )
    foreground_prior = result.get("foreground_prior", cfg.foreground_prior)
    if foreground_prior is None:
        foreground_prior = estimate_foreground_prior(train_samples, cfg)
    model = build_model(cfg, foreground_prior, device)
    visualise_predictions(
        model,
        selected_loader,
        checkpoint_path=Path(result["checkpoint_path"]),
        cfg=cfg,
        device=device,
        out_path=Path(cfg.output_dir) / f"positive_predictions_ksdd1_{result.get('strategy', cfg.strategy)}.png",
        n=len(selected_samples),
        thr=best_thr["thr"],
        min_area=best_thr.get("min_area", 0),
    )


## Build Training Config
Create a reusable config builder so the final run cell can choose a side-tuning strategy with one simple assignment.


In [ ]:
def make_train_config(strategy):
    if strategy not in AVAILABLE_STRATEGIES:
        raise ValueError(f"Unknown strategy={strategy!r}. Choose one of: {list(AVAILABLE_STRATEGIES)}")
    return TrainConfig(
        output_dir=str(OUTPUT_DIR),
        data_root=None,
        drive_data_root=str(DRIVE_DATA_ROOT),
        local_data_root=str(LOCAL_DATA_ROOT),
        copy_data_to_local=COPY_DATA_TO_LOCAL,
        use_preprocessed_train=USE_PREPROCESSED_TRAIN,
        preprocess_cache_root=str(PREPROCESS_CACHE_ROOT),
        drive_preprocess_cache_zip=str(DRIVE_PREPROCESS_CACHE_ZIP),
        copy_preprocess_cache_from_drive=COPY_PREPROCESS_CACHE_FROM_DRIVE,
        augmented_copies_per_sample=AUGMENTED_COPIES_PER_SAMPLE,
        cache_copies_per_epoch=CACHE_COPIES_PER_EPOCH,
        force_rebuild_preprocess_cache=FORCE_REBUILD_PREPROCESS_CACHE,
        model_name=MODEL_NAME,
        strategy=strategy,
        base_checkpoint_path=BASE_CHECKPOINT_PATH,
        freeze_base_model=FREEZE_BASE_MODEL,
        side_width=SIDE_WIDTH,
        side_dropout=SIDE_DROPOUT,
        side_init_mix=SIDE_INIT_MIX,
        side_lr_mult=SIDE_LR_MULT,
        mix_lr_mult=MIX_LR_MULT,
        base_lr_mult=BASE_LR_MULT,
        seed=SEED,
        epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        eval_batch_size=EVAL_BATCH_SIZE,
        image_size=IMG_SIZE,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        grad_clip_norm=GRAD_CLIP_NORM,
        threshold=THR,
        default_min_area=DEFAULT_MIN_AREA,
        require_cuda=REQUIRE_CUDA,
        selection_metric=SELECTION_METRIC,
        scheduler_metric=SCHEDULER_METRIC,
        image_fpr_penalty=IMAGE_FPR_PENALTY,
        threshold_pixel_fpr_lambda=PIXEL_FPR_PENALTY,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
        lr_scheduler=LR_SCHEDULER,
        cosine_t_max=COSINE_T_MAX,
        plateau_factor=PLATEAU_FACTOR,
        plateau_patience=PLATEAU_PATIENCE,
        plateau_min_lr=PLATEAU_MIN_LR,
        track_train_metrics=TRACK_TRAIN_METRICS,
        eval_train_metrics=EVAL_TRAIN_METRICS,
        sync_timing=SYNC_TIMING,
        run_threshold_sweep=RUN_THRESHOLD_SWEEP,
        threshold_sweep=THRESHOLD_SWEEP,
        threshold_min_areas=THRESHOLD_MIN_AREAS,
        num_workers=NUM_WORKERS,
        elastic_prob=ELASTIC_PROB,
        elastic_alpha=ELASTIC_ALPHA,
        elastic_sigma=ELASTIC_SIGMA,
        gaussian_noise_prob=GAUSSIAN_NOISE_PROB,
        gaussian_noise_std=GAUSSIAN_NOISE_STD,
    )


## Run Selected Strategy
Set the side-tuning strategy here, then train it, save strategy-specific artifacts, run the threshold sweep, and create positive-case previews.


In [ ]:
strategy = "side_tuning"

cfg = make_train_config(strategy)
history, result, model, val_loader, best_thr = run_experiment(cfg)
visualise_positive_predictions(cfg, result, best_thr, split="test", n=4)


## Display Saved Figures
Show the learning curves and prediction previews for the side-tuning strategy that just ran.


In [ ]:
from IPython.display import Image, display
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
strategy_name = result.get("strategy", cfg.strategy)
for file_name in [
    f"learning_curves_ksdd1_{strategy_name}.png",
    f"predictions_ksdd1_{strategy_name}.png",
    f"positive_predictions_ksdd1_{strategy_name}.png",
]:
    path = output_dir / file_name
    if path.exists():
        print(path)
        display(Image(filename=str(path)))
    else:
        print("Missing:", path)


## Inspect Saved JSON
Print the split summary, best result, and threshold sweep JSON for the current side-tuning strategy.


In [ ]:
import json
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
strategy_name = result.get("strategy", cfg.strategy)
for file_name in [
    "split_summary.json",
    f"best_result_{strategy_name}.json",
    f"threshold_sweep_ksdd1_{strategy_name}.json",
]:
    path = output_dir / file_name
    if path.exists():
        print()
        print("=" * 80)
        print(path)
        with open(path, "r", encoding="utf-8") as handle:
            print(json.dumps(json.load(handle), indent=2)[:4000])
    else:
        print("Missing:", path)


## Export Results To Drive
Save the run artifacts to Google Drive under `results/<strategy>/` for comparison across notebooks.


In [ ]:
import csv
import json
import shutil
from pathlib import Path


def _clean_csv_value(value):
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (list, tuple, dict)):
        return json.dumps(value)
    return value


def _write_csv(path, rows):
    rows = [dict(row) for row in rows] if rows else []
    fieldnames = []
    for row in rows:
        for key in row.keys():
            if key not in fieldnames:
                fieldnames.append(key)
    with open(path, "w", newline="", encoding="utf-8") as handle:
        if not fieldnames:
            return
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: _clean_csv_value(row.get(key, "")) for key in fieldnames})


def _history_rows(history_dict):
    lengths = [len(value) for value in history_dict.values() if isinstance(value, list)]
    if not lengths:
        return []
    rows = []
    for idx in range(max(lengths)):
        row = {"epoch": idx + 1}
        for key, values in history_dict.items():
            if isinstance(values, list) and idx < len(values):
                row[key] = values[idx]
        rows.append(row)
    return rows


def _as_path(value):
    if value is None:
        return None
    text = str(value)
    if not text:
        return None
    return Path(text)


def _copy_first_existing(target_dir, target_name, candidates):
    for candidate in candidates:
        path = _as_path(candidate)
        if path is not None and path.is_file():
            destination = target_dir / target_name
            shutil.copy2(path, destination)
            return destination
    return None


def _threshold_rows_and_best():
    rows = globals().get("threshold_results")
    best = globals().get("best_thr")
    if rows is not None:
        return list(rows), best

    threshold_path = None
    if "result" in globals() and isinstance(result, dict):
        threshold_path = result.get("threshold_sweep_path")
    path = _as_path(threshold_path)
    if path is not None and path.is_file():
        with open(path, "r", encoding="utf-8") as handle:
            payload = json.load(handle)
        if isinstance(payload, dict):
            return list(payload.get("rows", [])), payload.get("best", best)
        if isinstance(payload, list):
            return payload, best
    return [], best


strategy_name = "unknown"
if "result" in globals() and isinstance(result, dict):
    strategy_name = str(result.get("strategy", strategy_name))
elif "strategy" in globals():
    strategy_name = str(strategy)

output_dir = Path(".")
if "cfg" in globals() and hasattr(cfg, "output_dir"):
    output_dir = Path(cfg.output_dir)
elif "OUTPUT_DIR" in globals():
    output_dir = Path(OUTPUT_DIR)

save_dir = Path("/content/drive/MyDrive/results") / strategy_name
save_dir.mkdir(parents=True, exist_ok=True)

history_rows = _history_rows(history if "history" in globals() else {})
_write_csv(save_dir / "epochs.csv", history_rows)

threshold_rows, threshold_best = _threshold_rows_and_best()
_write_csv(save_dir / "threshold_area_sweep.csv", threshold_rows)
if threshold_best is not None:
    _write_csv(save_dir / "best_threshold_area.csv", [threshold_best])

if "result" in globals() and isinstance(result, dict):
    with open(save_dir / "best_result.json", "w", encoding="utf-8") as handle:
        json.dump(result, handle, indent=2)
    _write_csv(save_dir / "best_result.csv", [result])

learning_curves_path = result.get("learning_curves_path") if "result" in globals() and isinstance(result, dict) else None
predictions_path = result.get("predictions_path") if "result" in globals() and isinstance(result, dict) else None

copied_files = []
for copied in [
    _copy_first_existing(
        save_dir,
        "learning_curves.png",
        [
            learning_curves_path,
            output_dir / f"learning_curves_ksdd1_{strategy_name}.png",
            Path(f"learning_curves_ksdd1_{strategy_name}.png"),
            Path("learning_curves_ksdd1.png"),
        ],
    ),
    _copy_first_existing(
        save_dir,
        "predictions.png",
        [
            predictions_path,
            output_dir / f"predictions_ksdd1_{strategy_name}.png",
            Path(f"predictions_ksdd1_{strategy_name}.png"),
        ],
    ),
    _copy_first_existing(
        save_dir,
        "positive_predictions.png",
        [
            output_dir / f"positive_predictions_ksdd1_{strategy_name}.png",
            Path(f"positive_predictions_ksdd1_{strategy_name}.png"),
            Path("positive_predictions_ksdd1.png"),
            Path("predictions_ksdd1.png"),
        ],
    ),
]:
    if copied is not None:
        copied_files.append(str(copied))

print(f"Saved results to: {save_dir}")
print("CSV files:")
for name in ["epochs.csv", "threshold_area_sweep.csv", "best_threshold_area.csv", "best_result.csv"]:
    path = save_dir / name
    if path.exists():
        print(path)
print("PNG files:")
for path in copied_files:
    print(path)
